In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import sys

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

JSON_ROOT = Path("/Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/CT json")
REWEIGHTING_ROOT = Path("/Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/reweighting")

In [2]:
df = pd.read_csv(data_root / 'CBOS_data_to_weight_corrected.csv', low_memory=False)
print(f"Loaded: {df.shape[0]:,} obs × {df.shape[1]} cols, {df['survey_file'].nunique()} surveys")

df['teryt_id_VOIV'] = df['teryt_id_VOIV'].astype(str).str.zfill(7)

# Assign macroregions to the dataframe
cbos_macroregions = {1: ['1400000', '1000000'],
                     2: ['1200000', '2400000'],
                     3: ['0600000', '1800000', '2000000', '2600000'],
                     4: ['3000000', '3200000', '0800000'],
                     5: ['0200000', '1600000'],
                     6: ['2200000', '2800000', '0400000']
                     }

df['macroregion'] = df['teryt_id_VOIV'].apply(
    lambda x: next((int(mr) for mr, voivs in cbos_macroregions.items() if x in voivs), np.nan)
)

# For 2017 post-February, location_new has macroregion codes (1-6), not voivodeships
# Overwrite macroregion for those observations
mask_2017_post = (df['survey_year'] == 2017) & (df['survey_month'] > 2)
df.loc[mask_2017_post, 'macroregion'] = df.loc[mask_2017_post, 'location_new']

# Create household-level weight columns (start as copies of individual weights)
df['weight_VOIV_h']          = df['weight_VOIV']
df['weight_VOIV_500_h']      = df['weight_VOIV_500']
df['weight_VOIV_100_500_h']  = df['weight_VOIV_100_500']
df['weight_VOIV_100_h']      = df['weight_VOIV_100']

# Initialize columns for macroregion-level weights (start as copies of individual weights)
df['weight_MACRO']           = df['weight_VOIV']
df['weight_MACRO_500']       = df['weight_VOIV_500']
df['weight_MACRO_100_500']   = df['weight_VOIV_100_500']
df['weight_MACRO_100']       = df['weight_VOIV_100']

df['weight_MACRO_h']         = df['weight_VOIV_h']
df['weight_MACRO_500_h']     = df['weight_VOIV_500_h']
df['weight_MACRO_100_500_h'] = df['weight_VOIV_100_500_h']
df['weight_MACRO_100_h']     = df['weight_VOIV_100_h']

# Assign macroregion-level spatial groups (G_MACRO_500, G_MACRO_100_500, G_MACRO_100)
from regional_weighting import assign_macroregion_groups
META_MACRO_PATH = JSON_ROOT / "CBOS_U_cbos_macroregions_dict" / "META_CBOS_teryt_keys.json"
df = assign_macroregion_groups(df, META_MACRO_PATH)

print(f"Weight and group columns initialised. Shape: {df.shape}")
print("Years:", sorted(df['survey_year'].unique()))

# Verify macroregion group assignment
for c in ['G_MACRO_500', 'G_MACRO_100_500', 'G_MACRO_100']:
    n_valid = df[c].notna().sum()
    n_post99 = (df['survey_year'] >= 1999).sum()
    print(f"  {c}: {n_valid:,} assigned / {n_post99:,} post-1999 obs")

Loaded: 355,337 obs × 91 cols, 327 surveys


/Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/Code/tools/regional_weighting.py:334: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', 'Group 37', '

Weight and group columns initialised. Shape: (355337, 107)
Years: [np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
  G_MACRO_500: 232,663 assigned / 232,663 post-1999 obs
  G_MACRO_100_500: 232,663 assigned / 232,663 post-1999 obs
  G_MACRO_100: 232,663 assigned / 232,663 post-1999 obs


/Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/Code/tools/regional_weighting.py:334: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', 'Group 44', '

In [3]:
df

,Unnamed: 0,org_id,survey_file,survey_year,survey_month,age,year_born,sex,sex_L,city_size,...,weight_VOIV_100_500,weight_VOIV_100,G_VOIV_500,G_VOIV_100_500,G_VOIV_100,macroregion,weight_VOIV_h,weight_VOIV_500_h,weight_VOIV_100_500_h,weight_VOIV_100_h
0,0,1.0,CBOS_1_01_1990.sav,1990,1,33.0,1957.0,2.0,Kobieta,9.0,...,1.000000,1.000000,Group 106,Group 106,Group 106,NaN,1.000000,1.000000,1.000000,1.000000
1,1,2.0,CBOS_1_01_1990.sav,1990,1,40.0,1950.0,2.0,Kobieta,9.0,...,1.000000,1.000000,Group 106,Group 106,Group 106,NaN,1.000000,1.000000,1.000000,1.000000
2,2,3.0,CBOS_1_01_1990.sav,1990,1,58.0,1932.0,2.0,Kobieta,9.0,...,1.000000,1.000000,Group 9,Group 11,Group 11,NaN,1.000000,1.000000,1.000000,1.000000
3,3,4.0,CBOS_1_01_1990.sav,1990,1,44.0,1946.0,1.0,Mężczyzna,9.0,...,1.000000,1.000000,Group 9,Group 11,Group 11,NaN,1.000000,1.000000,1.000000,1.000000
4,7,8.0,CBOS_1_01_1990.sav,1990,1,64.0,1926.0,2.0,Kobieta,9.0,...,1.000000,1.000000,Group 3,Group 5,Group 5,NaN,1.000000,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355332,358520,921.0,CBOS_331_12_2017.sav,2017,12,60.0,1957.0,1.0,Mężczyzna,1.0,...,0.611965,0.611965,Group 69,Group 70,Group 70,NaN,0.611965,0.611965,0.611965,0.611965
355333,358521,922.0,CBOS_331_12_2017.sav,2017,12,36.0,1981.0,1.0,Mężczyzna,1.0,...,1.127158,1.127158,Group 69,Group 70,Group 70,NaN,1.127158,1.127158,1.127158,1.127158
355334,358522,923.0,CBOS_331_12_2017.sav,2017,12,41.0,1976.0,1.0,Mężczyzna,5.0,...,1.716070,1.716070,Group 69,Group 47,Group 48,NaN,1.716070,1.716070,1.716070,1.716070
355335,358523,924.0,CBOS_331_12_2017.sav,2017,12,46.0,1971.0,2.0,Kobieta,5.0,...,1.572886,1.572886,Group 69,Group 47,Group 48,NaN,1.572886,1.572886,1.572886,1.572886


In [5]:
# ============================================================
# Load CT JSON metadata (for validation)
# ============================================================
import json

old_voiv_path        = JSON_ROOT / "CBOS_old_voiv"        / "CBOS_all_years.json"
new_voiv_path        = JSON_ROOT / "CBOS_new_voiv"        / "CBOS_all_years.json"
old_groups_path      = JSON_ROOT / "CBOS_U_old_voiv_dict" / "CBOS_all_groups.json"
new_groups_path      = JSON_ROOT / "CBOS_U_new_voiv_dict" / "CBOS_all_groups.json"
macro_path           = JSON_ROOT / "CBOS_macroregions"    / "CBOS_all_years.json"
macro_groups_path    = JSON_ROOT / "CBOS_U_cbos_macroregions_dict" / "CBOS_all_groups.json"

with open(old_voiv_path,   encoding='utf-8') as f: ct_old_voiv   = json.load(f)
with open(new_voiv_path,   encoding='utf-8') as f: ct_new_voiv   = json.load(f)
with open(old_groups_path, encoding='utf-8') as f: ct_old_groups = json.load(f)
with open(new_groups_path, encoding='utf-8') as f: ct_new_groups = json.load(f)
with open(macro_path,      encoding='utf-8') as f: ct_macro      = json.load(f)
with open(macro_groups_path, encoding='utf-8') as f: ct_macro_groups = json.load(f)

print("Old voiv years:", list(ct_old_voiv.keys()))
print("New voiv years:", list(ct_new_voiv.keys()))
print("Macroregion years:", list(ct_macro.keys()))
print(f"Old groups: {len(ct_old_groups)}")
print(f"New groups: {len(ct_new_groups)}")
print(f"Macro groups: {len(ct_macro_groups)}")


Old voiv years: ['1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000']
New voiv years: ['1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017']
Macroregion years: ['1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017']
Old groups: 125
New groups: 291
Macro groups: 234


In [3]:
# ============================================================
# Run raking for all weight columns
# ============================================================
from regional_weighting import compute_regional_weights, compute_macroregion_weights, normalize_weights

# 1. Voivodeship-level raking (with macroregion fallback for 2017)
print("=" * 60)
print("VOIVODESHIP-LEVEL RAKING")
print("=" * 60)
df = compute_regional_weights(
    df,
    json_root=JSON_ROOT,
    max_iter=100,
    tol=1e-6,
    verbose=True,
)

# 2. Macroregion-level raking (1999-2017)
print("\n" + "=" * 60)
print("MACROREGION-LEVEL RAKING")
print("=" * 60)
df = compute_macroregion_weights(
    df,
    json_root=JSON_ROOT,
    max_iter=100,
    tol=1e-6,
    verbose=True,
)

# 3. Normalise all weight columns (mean = 1.0 per year)
all_weight_cols = [
    'weight_VOIV', 'weight_VOIV_h',
    'weight_VOIV_500', 'weight_VOIV_500_h',
    'weight_VOIV_100_500', 'weight_VOIV_100_500_h',
    'weight_VOIV_100', 'weight_VOIV_100_h',
    'weight_MACRO', 'weight_MACRO_h',
    'weight_MACRO_500', 'weight_MACRO_500_h',
    'weight_MACRO_100_500', 'weight_MACRO_100_500_h',
    'weight_MACRO_100', 'weight_MACRO_100_h',
]
df = normalize_weights(df, all_weight_cols, group_col='survey_year')

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
for col in all_weight_cols:
    s = df[col]
    sn = df[f'{col}_NORM']
    print(f"  {col:<28}: mean={s.mean():.4f}  min={s.min():.4f}  max={s.max():.4f}  |  NORM mean={sn.mean():.4f}")


VOIVODESHIP-LEVEL RAKING
Loading CT JSON files …

=== weight_VOIV / weight_VOIV_h ===
  Ind 1990: done  (mean=2472.8870, min=0.0000, max=123027.3795)
  Ind 1991: done  (mean=2983.7517, min=0.0000, max=119472.5181)
  Ind 1992: done  (mean=4332.7727, min=0.0000, max=111619.2099)
  Ind 1993: done  (mean=2840.6727, min=0.0000, max=88833.9840)
  Ind 1994: done  (mean=3038.6195, min=0.0000, max=90834.2424)
  Ind 1995: done  (mean=3444.0944, min=0.0000, max=116317.8298)
  Ind 1996: done  (mean=3517.7401, min=0.0000, max=144629.5673)
  Ind 1997: done  (mean=3297.5860, min=0.0000, max=105926.6566)
  Ind 1998: done  (mean=3521.0925, min=0.0000, max=153590.5717)
  Ind 1999: done  (mean=2924.4881, min=78.5961, max=54720.5165)
  Ind 2000: done  (mean=2901.9812, min=9.4390, max=76815.7642)
  Ind 2001: done  (mean=3064.8554, min=76.0123, max=46804.3968)
  Ind 2002: done  (mean=3025.4911, min=8.9356, max=48765.1392)
  Ind 2003: done  (mean=2896.0566, min=0.0042, max=47640.2114)
  Ind 2004: done  (mean

In [4]:
# Quick summary of weight columns
all_weight_cols = [c for c in df.columns if c.startswith('weight_') and not c.endswith('_NORM')]
print(f"{'Column':<35} {'mean':>8} {'min':>8} {'max':>8} {'NaN':>5}")
print("-" * 70)
for col in sorted(all_weight_cols):
    s = df[col]
    print(f"{col:<35} {s.mean():>8.4f} {s.min():>8.4f} {s.max():>8.4f} {s.isna().sum():>5}")

print(f"\n{'Column':<35} {'mean':>8} {'min':>8} {'max':>8}")
print("-" * 60)
norm_cols = [c for c in df.columns if c.endswith('_NORM')]
for col in sorted(norm_cols):
    s = df[col]
    print(f"{col:<35} {s.mean():>8.4f} {s.min():>8.4f} {s.max():>8.4f}")

Column                                  mean      min      max   NaN
----------------------------------------------------------------------
weight_MACRO                        2008.4468   0.0000 64766.4796     0
weight_MACRO_100                    2023.0908   0.0000 66962.7424     0
weight_MACRO_100_500                1949.0743   0.0000 66962.7424     0
weight_MACRO_100_500_h              713.1656   0.0000 32290.7382     0
weight_MACRO_100_h                  713.1656   0.0000 32290.7382     0
weight_MACRO_500                    2012.3668   0.0000 72145.1654     0
weight_MACRO_500_h                  713.1656   0.0000 33458.9707     0
weight_MACRO_h                      713.1656   0.0000 45858.9188     0
weight_VOIV                         3121.4372   0.0000 153590.5717     0
weight_VOIV_100                     3031.7261   0.0000 144629.5673     0
weight_VOIV_100_500                 2991.5148   0.0000 144629.5673     0
weight_VOIV_100_500_h               1040.8300   0.0000 49335.6867    

In [11]:
# ============================================================
# Validation: check that weighted marginals match CT targets
# ============================================================
from regional_weighting import CS_PROB_TO_POP_CLASS, EDUC_1990_REMAP

def validate_voiv(df, year, voiv_name, is_old=True):
    """Validate voivodeship-level raking for one voivodeship × year."""
    src = ct_old_voiv if is_old else ct_new_voiv
    loc_col   = 'location_old_L' if is_old else 'location_new_L'
    age_col   = 'age_1990_L'     if is_old else 'age_2000_L'
    educ_col  = 'educ_1990_L'    if is_old else 'educ_2000_L'
    hh_col    = 'hh_size_1990_L' if is_old else 'hh_size_2000_L'
    age_key   = 'E_age_sex_1990' if is_old else 'E_age_sex_2000'
    educ_key  = 'E_educ_sex_1990' if is_old else 'E_educ_sex_2000'
    hh_key    = 'E_hh_size_1990' if is_old else 'E_hh_size_2000'

    try:
        ct = src[str(year)][voiv_name][str(year)]
    except KeyError:
        print(f"No CT for {voiv_name} {year}"); return

    sub = df[(df['survey_year'] == year) & (df[loc_col] == voiv_name)].copy()
    if len(sub) == 0:
        print(f"No observations for {voiv_name} {year}"); return

    print(f"\n{'='*60}")
    print(f"VOIVODESHIP: {voiv_name}  Year: {year}  N={len(sub)}")
    print(f"{'='*60}")

    # age × sex
    print("\n[Individual] age × sex — weighted vs target:")
    age_sex_ct = ct.get(age_key, {})
    for key in sorted(age_sex_ct):
        if 'ogółem' in key: continue
        age_lbl, sex_json = [s.strip() for s in key.split('×', 1)]
        sex_df = 'Kobieta' if sex_json == 'kobiety' else 'Mężczyzna'
        mask = (sub[age_col] == age_lbl) & (sub['sex_L'] == sex_df)
        w_sum = sub.loc[mask, 'weight_VOIV'].sum()
        target = age_sex_ct[key]
        if target > 0:
            print(f"  {key:<45} weighted={w_sum:>12.1f}  target={target:>12.1f}  ratio={w_sum/target:.4f}")

    # educ × sex
    print("\n[Individual] educ × sex — weighted vs target:")
    educ_sex_ct = ct.get(educ_key, {})
    educ_s = sub[educ_col].map(lambda x: EDUC_1990_REMAP.get(x, x)) if is_old else sub[educ_col]
    for key in sorted(educ_sex_ct):
        if 'ogółem' in key: continue
        educ_lbl, sex_json = [s.strip() for s in key.split('×', 1)]
        sex_df = 'Kobieta' if sex_json == 'kobiety' else 'Mężczyzna'
        mask = (educ_s == educ_lbl) & (sub['sex_L'] == sex_df)
        w_sum = sub.loc[mask, 'weight_VOIV'].sum()
        target = educ_sex_ct[key]
        if target > 0:
            print(f"  {key:<55} weighted={w_sum:>12.1f}  target={target:>12.1f}  ratio={w_sum/target:.4f}")

    # hh_size (household)
    print("\n[Household] hh_size — weighted vs target:")
    hh_ct = ct.get(hh_key, {})
    for hh_lbl, target in hh_ct.items():
        if hh_lbl == 'ogółem' or target <= 0: continue
        mask = sub[hh_col] == hh_lbl
        w_sum = sub.loc[mask, 'weight_VOIV_h'].sum()
        print(f"  {hh_lbl:<25} weighted={w_sum:>12.1f}  target={target:>12.1f}  ratio={w_sum/target:.4f}")

    # pop_class
    print("\n[Individual] pop_class — weighted vs target:")
    pc_ct = ct.get('pop_class', {})
    for cs_val in sorted(sub['cs_prob'].dropna().unique()):
        keys = CS_PROB_TO_POP_CLASS.get(float(cs_val), [])
        target = sum(pc_ct.get(k, 0.0) for k in keys)
        mask = sub['cs_prob'] == cs_val
        w_sum = sub.loc[mask, 'weight_VOIV'].sum()
        label = '+'.join(keys) if keys else str(cs_val)
        line = f"  cs_prob={cs_val}  {label[:40]:<40} weighted={w_sum:>12.1f}  target={target:>12.1f}"
        print(line + (f"  ratio={w_sum/target:.4f}" if target > 0 else "  target=0"))


def validate_macro(df, year, macro_id):
    """Validate macroregion-level raking for one macroregion × year."""
    try:
        ct = ct_macro[str(year)][str(macro_id)][str(year)]
    except KeyError:
        print(f"No CT for macroregion {macro_id} year {year}"); return

    sub = df[(df['survey_year'] == year) & (df['macroregion'] == macro_id)].copy()
    if len(sub) == 0:
        print(f"No observations for macroregion {macro_id} year {year}"); return

    print(f"\n{'='*60}")
    print(f"MACROREGION: {macro_id}  Year: {year}  N={len(sub)}")
    print(f"{'='*60}")

    age_col = 'age_2000_L'
    educ_col = 'educ_2000_L'
    hh_col = 'hh_size_2000_L'

    # age × sex — macroregion individual weights
    print("\n[Individual] age × sex — weighted vs target (weight_MACRO):")
    age_sex_ct = ct.get('E_age_sex_2000', {})
    for key in sorted(age_sex_ct):
        if 'ogółem' in key: continue
        age_lbl, sex_json = [s.strip() for s in key.split('×', 1)]
        sex_df = 'Kobieta' if sex_json == 'kobiety' else 'Mężczyzna'
        mask = (sub[age_col] == age_lbl) & (sub['sex_L'] == sex_df)
        w_sum = sub.loc[mask, 'weight_MACRO'].sum()
        target = age_sex_ct[key]
        if target > 0:
            print(f"  {key:<45} weighted={w_sum:>12.1f}  target={target:>12.1f}  ratio={w_sum/target:.4f}")

    # hh_size
    print("\n[Household] hh_size — weighted vs target (weight_MACRO_h):")
    hh_ct = ct.get('E_hh_size_2000', {})
    for hh_lbl, target in hh_ct.items():
        if hh_lbl == 'ogółem' or target <= 0: continue
        mask = sub[hh_col] == hh_lbl
        w_sum = sub.loc[mask, 'weight_MACRO_h'].sum()
        print(f"  {hh_lbl:<25} weighted={w_sum:>12.1f}  target={target:>12.1f}  ratio={w_sum/target:.4f}")


# Run validation
validate_voiv(df, year=1995, voiv_name='bydgoskie', is_old=True)
validate_voiv(df, year=2005, voiv_name='małopolskie', is_old=False)
validate_macro(df, year=2005, macro_id=1)
validate_macro(df, year=2017, macro_id=3)

# Also show that 2017 VOIV weights used macroregion CT
print(f"\n{'='*60}")
print("2017 VOIV weights (via macroregion fallback):")
for macro_id in sorted(df.loc[df['survey_year']==2017, 'macroregion'].dropna().unique()):
    sub = df[(df['survey_year']==2017) & (df['macroregion']==macro_id)]
    print(f"  Macroregion {int(macro_id)}: N={len(sub)}, "
          f"weight_VOIV mean={sub['weight_VOIV'].mean():.4f}, "
          f"weight_MACRO mean={sub['weight_MACRO'].mean():.4f}")



VOIVODESHIP: bydgoskie  Year: 1995  N=463

[Individual] age × sex — weighted vs target:
  0-9 × kobiety                                 weighted=         0.0  target=     79214.0  ratio=0.0000
  0-9 × mężczyźni                               weighted=         0.0  target=     82942.0  ratio=0.0000
  10-19 × kobiety                               weighted=    156419.8  target=     98174.0  ratio=1.5933
  10-19 × mężczyźni                             weighted=    163329.4  target=    102550.0  ratio=1.5927
  20-29 × kobiety                               weighted=    126867.5  target=     79626.0  ratio=1.5933
  20-29 × mężczyźni                             weighted=    129844.9  target=     81526.0  ratio=1.5927
  30-39 × kobiety                               weighted=    137606.3  target=     86366.0  ratio=1.5933
  30-39 × mężczyźni                             weighted=    137728.7  target=     86476.0  ratio=1.5927
  40-49 × kobiety                               weighted=    146522.3  

In [8]:
# ============================================================
# Save results
# ============================================================
output_path = REWEIGHTING_ROOT / 'CBOS_data_reweighted.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved {df.shape[0]:,} rows × {df.shape[1]} cols → {output_path}")

# Summary of all weight columns
weight_cols = [c for c in df.columns if c.startswith('weight_')]
print(f"\n{len(weight_cols)} weight columns total:")
for col in sorted(weight_cols):
    s = df[col]
    print(f"  {col:<35}: mean={s.mean():.4f}  min={s.min():.4f}  max={s.max():.4f}  NaN={s.isna().sum()}")


Saved 355,337 rows × 123 cols → /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/reweighting/CBOS_data_reweighted.csv

32 weight columns total:
  weight_MACRO                       : mean=2008.4468  min=0.0000  max=64766.4796  NaN=0
  weight_MACRO_100                   : mean=2023.0908  min=0.0000  max=66962.7424  NaN=0
  weight_MACRO_100_500               : mean=1949.0743  min=0.0000  max=66962.7424  NaN=0
  weight_MACRO_100_500_NORM          : mean=1.0000  min=0.0000  max=24.2519  NaN=0
  weight_MACRO_100_500_h             : mean=713.1656  min=0.0000  max=32290.7382  NaN=0
  weight_MACRO_100_500_h_NORM        : mean=1.0000  min=0.0000  max=31.6372  NaN=0
  weight_MACRO_100_NORM              : mean=1.0000  min=0.0000  max=23.3614  NaN=0
  weight_MACRO_100_h                 : mean=713.1656  min=0.0000  max=32290.7382  NaN=0
  weight_MACRO_100_h_NORM            : mean=1.0000  min=0.0000  max=31.6372  NaN=0
  weight_MACR